Note: I recommend running each panel separately on a fresh kernel if memory is a concern

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "utils").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT))

import importlib
import os
from datetime import UTC, datetime

import cartopy.crs as ccrs
import numpy as np
import xarray as xr
from dotenv import find_dotenv, load_dotenv

from utils import analysis_utils, isku_utils, plotting_utils, stats_utils

importlib.reload(analysis_utils)
importlib.reload(isku_utils)
importlib.reload(plotting_utils)

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<module 'utils.plotting_utils' from '/home/emily_zuetell/projects/poreallas/utils/plotting_utils.py'>

In [2]:
load_dotenv(
    find_dotenv("october_local.env", raise_error_if_not_found=True), override=True
)

EFFECTS_URI = os.environ["POREALLAS_EFFECTS_URI"]
IMPACT_REGION_POLYGONS = os.environ["POREALLAS_REGIONS_POLYGONS_URI"]
SOCIOECONOMICS_URI = os.environ["POREALLAS_SOCIOECONOMICS_URI"]

# Climate Data
ERA5_URI = os.environ["POREALLAS_ERA5_URI"]

In [3]:
# Baseline period for Impact
effect = xr.open_datatree(EFFECTS_URI, consolidated=False)
BASELINE_PERIOD = analysis_utils.get_baseline_period(effect, years=30)
# Define Forecast Months
FC_MONTHS = [10, 11, 12, 1, 2, 3]

config = analysis_utils.ImpactConfig(
    version=f"v{datetime.now(tz=UTC):%y%m%d}",
    baseline_period=BASELINE_PERIOD,
    polygons_path=os.environ["POREALLAS_REGIONS_POLYGONS_URI"],
    socioeconomics_path=os.environ["POREALLAS_SOCIOECONOMICS_URI"],
    rate=True,
    age_weight=True,
    months=FC_MONTHS,
    hotonly="hotonly",
    dims=["number", "sample"],
)

In [4]:
### Load Climate Data ###
forecast = xr.open_zarr(
    "../data/v2610/v20261006_parsed_forecast.zarr"
)  # Daily ensemble member temp
hindcast = xr.open_zarr(
    "../data/v2610/v20261006_parsed_hindcast.zarr"
)  # Daily ensemble mean temp

# BASELINE
era5 = xr.open_zarr(ERA5_URI).sel(time=slice("1996-01-01", "2025-12-31")) - 273.15

## Fig 1a: Forecast Anomaly Map

In [5]:
## Fig 1a ##
## Forecast Anomaly Map
forecast_month = (
    forecast.groupby("time.month").mean().sel(month=config.months)
)  # (number, month, lat, lon) 6 config monthsd
hindcast_month = hindcast.groupby("time.month").mean()  # (month, lat, lon), 12 months
fc_anomaly = (
    forecast_month["tas"] - hindcast_month["tas"]
)  # aligns on month, broadcasts over number
fc_land = analysis_utils.land_only(
    fc_anomaly, lat_name="lat", lon_name="lon", clip_antarctica=False
)
anom = fc_land.mean(
    dim=["month"]
)  # (number, lat, lon) — 6-month (Oct–Mar) mean anomaly
ens_mean = anom.mean("number")  # (lat, lon)

# plot prep
mask = stats_utils.compute_agreement_mask(anom, threshold=0.8, member_dim="number")

In [6]:
def save_panel_a(path, ens_mean, mask):
    ds = xr.Dataset({"ens_mean": ens_mean, "agreement": mask}).reset_coords(drop=True)
    ds.attrs = {
        "version": config.version,
        "fc_months": list(config.months),
        "agreement_threshold": 0.8,
        "units": "degC anomaly vs hindcast climatology",
    }
    ds.drop_encoding().chunk(-1).to_zarr(path, mode="w", consolidated=True)

In [7]:
save_panel_a(
    f"../data/figures/{config.version}_fig1a.zarr",
    ens_mean, mask
)

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


## Fig 1b: Method Diagram

### Helper Functions

In [8]:
def _flat(da):
    # Flatten and check xarray
    v = np.asarray(da).ravel()
    return v[np.isfinite(v)]


## Climate Data Processing ##
def select_climate(forecast, reanalysis, region, month=None, pad=3):
    # Subset climate data
    fc = forecast.sel(region=region)
    era5 = reanalysis.sel(region=region)

    # Optionally select a single month
    if month is not None:
        era5 = era5.where(era5.month.isin([month]).compute(), drop=True)
        fc = fc.sel(time=fc.time.dt.month == month)

    vals = np.concatenate([_flat(fc), _flat(era5)])
    return {
        "forecast": fc,
        "era5": era5,
        "n_members": fc.sizes["number"],
        "n_years": np.unique(era5.time.dt.year).size,
        "xlim": (vals.min() - pad, vals.max() + pad),
    }

### Data Pre-Processing

In [9]:
## Fig 1b ##

forecast_ir = isku_utils.grid_to_ir(
    forecast - 273.15, savefile=None
)  # (number, time, region), °C
era5_ir = isku_utils.grid_to_ir(era5, savefile=None)  # (time, region), °C
era5_ir = era5_ir.assign_coords(month=era5_ir["time"].dt.month)

/home/emily_zuetell/projects/poreallas/utils/isku_utils.py:28: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  _region_weights = xr.load_dataset(uri)[
/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/zarr/core/group.py:3289: ZarrUserWarning: Object at zarr.json:Zone.Identifier is not recognized as a component of a Zarr hierarchy.
  warnings.warn(
/home/emily_zuetell/projects/poreallas/utils/isku_utils.py:28: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfull

In [10]:
def save_panel_b(
    path,
    forecast_ir,
    era5_ir,
    betas_mmt,
    version,
    beta_var="beta_hotonly",
    region="BWA.3",
    month=1,
    age_cohort="age65plus",
    title="Kgalagadi, Botswana",
):
    clim = select_climate(forecast_ir["tas"], era5_ir["tas"], region, month=month)
    b = (
        betas_mmt[beta_var]
        .sel(region=region, age_cohort=age_cohort)
        .transpose("sample", "tas_bin")
    )
    mmt = betas_mmt["mmt"].sel(region=region, age_cohort=age_cohort).mean("sample")
    years = era5_ir["time"].dt.year

    ds = xr.Dataset(
        {
            "fc_tas": clim["forecast"].rename(time="fc_time").reset_coords(drop=True),
            "era5_tas": clim["era5"].rename(time="era5_time").reset_coords(drop=True),
            "beta": b.reset_coords(drop=True),
            "mmt": mmt.reset_coords(drop=True),
        }
    )
    ds.attrs = {
        "version": version,
        "region": region,
        "region_title": title,
        "month": month,
        "age_cohort": age_cohort,
        "beta_var": beta_var,
        "era5_period": f"{int(years.min())}-{int(years.max())}",
    }
    ds.drop_encoding().chunk(-1).to_zarr(path, mode="w", consolidated=True)

In [11]:
save_panel_b(
    f"../data/figures/{config.version}_fig1b.zarr",
    forecast_ir, era5_ir, effect["/fixed_beta"], config.version,
    beta_var="beta_hotonly" if config.hotonly == "hotonly" else "beta",
)

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


## Fig 1c: Slope at Avg. Forecast

### Helper Functions ###

In [12]:
def merge_regions(ds, socioec_ds, forecast_ds):
    # Merge betas and socioeconomics
    if isinstance(ds, xr.DataTree):
        ds = ds.to_dataset()
    merged = xr.merge(
        [ds, socioec_ds[["pop"]], forecast_ds[["forecast_mean"]]],
        join="inner",
    )

    return merged


def compute_slope_at_forecast(merged):
    """
    Compute d(beta)/d(tas) evaluated at each region's forecast_mean,
    for every sample and age_cohort.
    """
    tas_bin = merged["tas_bin"].values

    def _slope_at_point(beta_curve, x0):
        # beta_curve: 1D array over tas_bin; x0: scalar forecast_mean for this region
        dbeta_dtas = np.gradient(beta_curve, tas_bin)
        return np.interp(x0, tas_bin, dbeta_dtas)

    slope = xr.apply_ufunc(
        _slope_at_point,
        merged["beta"],
        merged["forecast_mean"],
        input_core_dims=[["tas_bin"], []],
        output_core_dims=[[]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[merged["beta"].dtype],
    )

    return slope.rename("beta_slope_at_forecast")

In [13]:
forecast_ir = isku_utils.grid_to_ir(
    forecast - 273.15, savefile=None
)  # (number, time, region), shared with (b)
forecast_ds = (
    forecast_ir["tas"]
    .groupby("time.month")
    .mean()
    .sel(month=config.months)  # Select only 6-forecast months
    .mean(dim="number")
    .to_dataset(name="forecast_mean")  # (month, region)
)

ds1 = merge_regions(
    effect["/fixed_beta"], config.socioeconomics, forecast_ds
)  # inner join on region

slope = compute_slope_at_forecast(
    ds1.sel(age_cohort="age65plus").mean(dim="month")  # forecast_mean -> (region,)
)

_polygons_slope = analysis_utils.xarray_to_gpd(
    slope.mean(dim="sample"), config.polygons
)
gdf_rob = _polygons_slope.to_crs(ccrs.Robinson().proj4_init)  # (sample, region)

/home/emily_zuetell/projects/poreallas/utils/isku_utils.py:28: RuntimeWarning: Failed to open Zarr store with consolidated metadata, but successfully read with non-consolidated metadata. This is typically much slower for opening a dataset. To silence this warning, consider:
1. Consolidating metadata in this existing store with zarr.consolidate_metadata().
2. Explicitly setting consolidated=False, to avoid trying to read consolidate metadata, or
3. Explicitly setting consolidated=True, to raise an error in this case instead of falling back to try reading non-consolidated metadata.
  _region_weights = xr.load_dataset(uri)[
/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/zarr/core/group.py:3289: ZarrUserWarning: Object at zarr.json:Zone.Identifier is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


In [14]:
def save_panel_c(path, slope, version, months, age_cohort="age65plus"):
    ds = (
        slope.mean(dim="sample")
        .to_dataset(name="beta_slope_at_forecast")
        .reset_coords(drop=True)
    )
    ds.attrs = {
        "version": config.version,
        "age_cohort": "age65plus",
        "beta_var": "beta",  # what compute_slope_at_forecast actually uses
        "months_averaged": [int(m) for m in ds1["month"].values],
        "units": "rate / degC",
    }
    ds.drop_encoding().chunk(-1).to_zarr(path, mode="w", consolidated=True)

In [15]:
save_panel_c(
    f"../data/figures/{config.version}_fig1c.zarr",
    slope, config.version, ds1["month"].values,
)

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=35, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/zarr/api/asynchronous.py:231: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
